# Genetically anchoring the shared TBI–AD microglial axis

**Reproducible walkthrough of the genetic-anchoring arm.** This notebook reruns the
*analysis* layers from the committed result tables in `results/`. The three heavy
fetch/compute steps (GWAS stream, ChromBPNet inference, S-LDSC LD-score computation)
are documented with the exact commands but not re-executed here — see `DATA_PROVENANCE.md`
and `code/` for those. Everything below runs in the **analysis** environment
(`requirements.txt`, ENV 1) in seconds.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 120)


## Layer 0 — the anchor gene set
38 human axis genes in three modules. Accelerator (21) is the test annotation;
resolution (9) is the pre-specified negative control; DAM-only (8) is a secondary set.


In [ ]:
targets = pd.read_csv('results/axis_targets.csv')
print(targets.arm.value_counts())
targets[['gene','arm','chrom','tss']].head(10)


## Layer 1 — AD-variant enrichment in microglial accelerator enhancers
AD variants (Bellenguez GCST90027158) intersected with Corces C24 microglial
enhancer peaks in axis cis-windows. Fisher + MAF-matched permutation, per arm.
**Headline: accelerator OR = 1.56, permutation P = 0.0022; resolution OR = 0.**


In [ ]:
enr = pd.read_csv('results/enrichment_results.csv')
enr


## Layer 2 — ChromBPNet allelic chromatin effect
Each AD variant scored for its predicted effect on microglial accessibility
(model reconstructed bias-free; validated vs Corces' own scores, |LFC| r=0.986).
**AD-associated accelerator variants disrupt chromatin more than non-associated
(Mann-Whitney P = 0.005).** Top chromatin-disruptors below.


In [ ]:
top = pd.read_csv('results/chrombpnet_top_variants.csv')
top[['rsid','axis_gene','axis_arm','p_value','lfc','jsd']].head(10)


In [ ]:
# arm-level: signal is in AD-associated variants, not the enhancer background
arm = pd.read_csv('results/chrombpnet_arm_summary.csv')
arm


## Layer 3 — S-LDSC partitioned heritability
Fraction of AD SNP-heritability in the annotation, conditional on baseline-LD v2.2.
**Microglial regulatory DNA (1.5% of genome) carries 31% of AD h²: 21× enrichment,
P = 1.1×10⁻⁵.** Within axis enhancers the coefficient is accelerator-positive,
resolution/DAM-negative (direction, not individual significance).


In [ ]:
sldsc = pd.read_csv('results/sldsc_results.csv')
sldsc[['label','prop_snps','prop_h2','enrichment','enrichment_p','coef_z']].round(4)


## Synthesis — the two arms converge on the same accelerator genes
Per accelerator gene: environmental installation (mean microglial log₂FC across
TBI + AD single-nucleus datasets) vs genetic risk load (AD-associated variants in
microglial enhancers). TREM2/APOE/TNF carry the genetic load (≥3 risk variants);
SPP1/GPNMB/CST7 are environmental-dominant effectors.


In [ ]:
conv = pd.read_csv('results/crossarm_convergence.csv')
conv['both_arm'] = (conv.n_AD_assoc>=3) & (conv[['AD_lfc','TBI_lfc']].mean(axis=1)>0)
conv.sort_values('n_AD_assoc', ascending=False)[['gene','AD_lfc','TBI_lfc','n_AD_assoc','max_chrombpnet_jsd','both_arm']].head(12)


## Figures
All five publication figures are in `figures/`. Rendered inline below.


In [ ]:
from matplotlib import image as mpimg
for f in ['gwas_axis_loci','enrichment_results','chrombpnet_results','sldsc_heritability','crossarm_synthesis']:
    im = mpimg.imread(f'figures/{f}.png')
    fig, ax = plt.subplots(figsize=(11, 11*im.shape[0]/im.shape[1]))
    ax.imshow(im); ax.axis('off'); ax.set_title(f, fontsize=9)
    plt.show()


---
**Thesis.** TBI installs environmentally what AD risk alleles predispose to
genetically, and both converge on the same microglial accelerator axis. See
`thesis.md` for the full argument and `DATA_PROVENANCE.md` for every data source.
